# 门控段与稠密段的协同结构

按第 9 章五步设计流程，把 Gated-KV SDPA 的计算图拆段、定归属、定交接。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 计算图分解

```text
段 0：门控分数准备（方案 A：Host / 方案 B：QK 轻量打分 —— 选做）
段 1：块选择（确定性排名 top-k）                    ← SIMT 倾向（第 9 章实践三的模式放大到块粒度）
段 2：选中块索引紧凑化 + 元数据           ← SIMT（按唯一 rank 写入，避免原子登记乱序）
段 3：选中块的 flash 注意力               ← Cube/Vector 交替（第 10 章循环的"K 块序列换成选中序列"）
```


## 2. 段归属与理由

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">段</th><th style="text-align:left;">归属</th><th style="text-align:left;">理由</th><th style="text-align:left;">复用</th></tr>
<tr><td>1 块选择（确定性排名）</td><td>SIMT</td><td>主线只有 16/32 块；每候选比较全部分数及块号得到唯一 rank，O(n²)，不依赖原子门槛猜测</td><td>09.05 实践三</td></tr>
<tr><td>2 索引紧凑化</td><td>SIMT</td><td>rank<k 时写 sel_idx[rank]，计数固定为 k；输出次序与 Golden 一致</td><td>09.05 实践一</td></tr>
<tr><td>3 flash 注意力</td><td>Cube + Vector</td><td>选中块集上跑第 10 章循环——块粒度保证段 3 数据规整</td><td>第 10 章全部</td></tr>
</table>

实现路径注记：段 1 的 top-k 若退化为"阈值初筛/top-1"且候选块能装进片上，可用 tile 级 `pl.gather` 比较模式或 `pl.argmax` 替代 SIMT 仲裁（见 09.03 选型对照）——教学主线保持 SIMT 版以训练仲裁技能；方案 B（QK 轻量打分）实现时两者都值得尝试并对比。

**为什么本课程练习 Device 选择**：用于掌握 Select-then-Compute 的实现。若分数原本在 Device，可减少回读；但主线分数由 Host 产生且仅 16/32 项，Device 选择未必更快；在这个主线场景中，分数本来就在 Host，不存在必须先做的分数 D2H 回读。是否能与 Q 准备重叠取决于资源与依赖。


**32 选 8 的最小选择实现**：本例独立于注意力 Kernel，显式单核启动；16 块或其他 ratio 时按 num_blocks/k 特化常量及形状。


In [ ]:
NUM_BLOCKS, TOP_K = 32, 8

@pl.vector_function(mode="simt", max_threads=32)
def select_rank(scores, selected):
    i = pl.simt.cast(pl.simt.linear_thread_idx(), pl.DT_INT32)
    if i < NUM_BLOCKS:
        own = scores[i]
        rank = 0
        for j in pl.range(0, NUM_BLOCKS, 1):
            other = scores[j]
            if other > own or (other == own and j < i):
                rank = rank + 1
        if rank < TOP_K:
            selected[rank] = i

@pl.jit()
def select_kernel(scores: pl.Tensor[[NUM_BLOCKS], pl.DT_FP32],
                  selected: pl.Tensor[[TOP_K], pl.DT_INT32]):
    with pl.section_vector():
        select_rank[32](scores, selected)

torch.manual_seed(0)
g_host = torch.randn(NUM_BLOCKS, dtype=torch.float32)
device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
g_device = g_host.to(device)
selected = torch.empty(TOP_K, dtype=torch.int32, device=device)
select_kernel[None, 1](g_device, selected)
torch.npu.synchronize()
expected = torch.argsort(g_host, descending=True, stable=True)[:TOP_K].to(torch.int32)
torch.testing.assert_close(selected.cpu(), expected, rtol=0, atol=0)

此实现只解决选择，不能据此声称第 11 章完整融合算子已实现。还须测并列/全同分/负分，并将输出发布给注意力段。


## 3. 交接设计（第 9 章三件套）

1. **容量分配**：本章 top-k 的 count 已由 Host 确定为 k，参考实现直接分配 k 个索引槽位；若扩展为返回数量不定的阈值筛选，再按 num_blocks 上界分配并返回 count；
2. **紧凑化**：选中块号连续存放（`sel_idx[0..count)`），**段 3 按 sel_idx 顺序遍历**——块地址计算变成查表（gather 模式）；
3. **元数据同步**：最初用独立 selection Kernel 在同一 Stream 完成后再启动注意力段；若融合到 MIX Kernel，需设计选择生产者到全部消费组的发布与等待，GM 直写不由 auto_mutex 自动覆盖；段 3 内 K/V 块的 load 坐标来自 sel_idx[kv_pos]（运行时索引——Tile load 的坐标是运行时表达式，天然支持）。


## 4. 段 3 的关键改动：K 块循环换"选中序列"

对照第 10 章走读版，flash 循环只改三处：

```python
for kv_pos in pl.range(0, sel_count, 1):          # 改 ①：循环上界 = 选中数
    kv = pl.getval(sel_idx, kv_pos)                          # 改 ②：块号查表（Gather）
    # load K/V 块的坐标 = kv * KV_BLOCK 起         # 改 ③：坐标由块号计算
    # s = Q @ K_kv^T → online 更新 → p @ V_kv 累加（其余与第 10 章一致）
```

改 ② 让注意力段按原始块号读取选中的 K/V——**ratio=25% 时，在无 mask 基线中只计算约 25% 的 KV token 对应的 QKᵀ/PV；整算子加速比仍须实测**。


### 从 32 选 8 扩到完整主线

完整参考 `src/gated_kv_reference.py` 用编译期 N/K 支持 RFC 的 16/32 候选和三档 ratio。每个候选计算严格排名，唯一写入 selected[rank]；K 已由 Host 规格确定，无需原子计数。选择 Kernel 仅启动一个 Vector 核，结果由所有 batch/Q 行共享。

消费者复用第 10 章 `prepare_attention`：原始块号通过 `pl.getval(selected,j)` 读出，再形成 K/V 的 GM 偏移。选择与注意力是同一 Stream 上的两个 Kernel，前者完整结束后消费者才能开始；attention 内部的四事件只管理本组 Cube/Vector 数据交接，不能拿来替代全局选择结果的发布。


## 5. 课后练习

1. （思考）分数原本在 Device 与本课程 Host 随机分数两种情况下，Device 选择的成本/收益有何不同？
2. （实践）画出三段的五步设计表（段/归属/交接载体/同步点/复用模块）——对照 09.03 的表格式。

In [ ]:
!cat ./answer/11.03_answer.txt

## 6. 本节小结

- 三段结构：**确定性 top-k（SIMT）→ 按 rank 写索引（SIMT）→ 选中块 flash 注意力（CV）**；
- 交接三件套齐用：上界分配、紧凑化、元数据同步；
- 段 3 对第 10 章只改三处——**查表定位选中块是稀疏注意力性价比的来源**。